In [1]:
import os
import pandas as pd
import numpy as np
import xarray as xr
from dask.distributed import Client, LocalCluster

# ============================
# User settings
# ============================
sdate, edate = '20090701', '20240630'
write_path = '/scratch/ng72/ms5578/solar_wind_tseries/'

# BARRA-R2 variable paths
rsds_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/rsds/latest/"
u_path    = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/ua50m/latest/"
v_path    = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/va50m/latest/"

# Generator site list
gen_csv = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv"

# Dask cluster

client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 7
Total threads: 28,Total memory: 125.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42261,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:40577,Total threads: 4
Dashboard: /proxy/41117/status,Memory: 17.88 GiB
Nanny: tcp://127.0.0.1:39321,


In [2]:
# ============================
# Functions
# ============================
def get_files(path, sdate, edate):
    """Get sorted list of files in date range."""
    fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
    fpaths = [s for s in os.listdir(path) if any(f in s for f in fdates)]
    fpaths = sorted([os.path.join(path, f) for f in fpaths])
    if not fpaths:
        raise FileNotFoundError(f"No NetCDF files found for {sdate} to {edate} in {path}")
    print(f"{path}: found {len(fpaths)} files")
    return fpaths


def extract_and_save(varname, fpaths, coords_df, out_file):
    """
    Extract timeseries for all sites in coords_df, add DUID column,
    and save all combined into one CSV file.
    """
    with xr.open_dataset(fpaths[0]) as ds:
        all_lats = ds['lat'].values
        all_lons = ds['lon'].values
        if np.any(all_lons > 180) and np.any(coords_df['lon'] < 0):
            coords_df['lon'] = coords_df['lon'] % 360

    combined_df = []

    for _, row in coords_df.iterrows():
        lat_idx = np.abs(all_lats - row['lat']).argmin()
        lon_idx = np.abs(all_lons - row['lon']).argmin()

        ds = xr.open_mfdataset(fpaths, combine='by_coords', chunks={'time': 50}, parallel=True, engine='h5netcdf')
        ts = ds[varname].isel(lat=lat_idx, lon=lon_idx).to_series().reset_index()
        ts['DUID'] = row['DUID']

        combined_df.append(ts)

    combined_df = pd.concat(combined_df, ignore_index=True)
    combined_df.to_csv(out_file, index=False)
    print(f"Saved combined {varname} data for all sites to {out_file}")


def extract_wind(u_paths, v_paths, coords_df, out_file):
    """
    Extract ua50m and va50m for all sites, combine into one DataFrame,
    add DUID column, and save as a single CSV file.
    """
    with xr.open_dataset(u_paths[0]) as ds:
        all_lats = ds['lat'].values
        all_lons = ds['lon'].values
        if np.any(all_lons > 180) and np.any(coords_df['lon'] < 0):
            coords_df['lon'] = coords_df['lon'] % 360

    combined_df = []

    for _, row in coords_df.iterrows():
        lat_idx = np.abs(all_lats - row['lat']).argmin()
        lon_idx = np.abs(all_lons - row['lon']).argmin()

        u_ds = xr.open_mfdataset(u_paths, combine='by_coords', chunks={'time': 50}, parallel=True, engine='h5netcdf')
        v_ds = xr.open_mfdataset(v_paths, combine='by_coords', chunks={'time': 50}, parallel=True, engine='h5netcdf')

        u_ts = u_ds['ua50m'].isel(lat=lat_idx, lon=lon_idx).to_series()
        v_ts = v_ds['va50m'].isel(lat=lat_idx, lon=lon_idx).to_series()

        df = pd.DataFrame({
            'time': u_ts.index,
            'ua50m': u_ts.values,
            'va50m': v_ts.values
        })

        df['DUID'] = row['DUID']
        combined_df.append(df)

    combined_df = pd.concat(combined_df, ignore_index=True)
    combined_df.to_csv(out_file, index=False)
    print(f"Saved combined wind data for all sites to {out_file}")


# ============================
# Main
# ============================
if __name__ == "__main__":
    gen_df = pd.read_csv(gen_csv)

    solar_df = gen_df[gen_df['fuel_source_primary'] == 'Solar'][['DUID', 'lat', 'lon']].copy()
    wind_df  = gen_df[gen_df['fuel_source_primary'] == 'Wind'][['DUID', 'lat', 'lon']].copy()

    rsds_files = get_files(rsds_path, sdate, edate)
    u_files    = get_files(u_path, sdate, edate)
    v_files    = get_files(v_path, sdate, edate)

    solar_out_file = os.path.join(write_path, "rsds_solar.csv")
    wind_out_file  = os.path.join(write_path, "wind.csv")

/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/rsds/latest/: found 180 files
/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/ua50m/latest/: found 180 files
/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/va50m/latest/: found 180 files


In [3]:
extract_and_save("rsds", rsds_files, solar_df, solar_out_file)
extract_wind(u_files, v_files, wind_df, wind_out_file)

Saved combined rsds data for all sites to /scratch/ng72/ms5578/solar_wind_tseries/rsds_solar.csv
Saved combined wind data for all sites to /scratch/ng72/ms5578/solar_wind_tseries/wind.csv


In [4]:
# import os
# import pandas as pd
# import numpy as np
# import xarray as xr
# from dask.distributed import Client, LocalCluster

# # ============================
# # User settings
# # ============================
# sdate, edate = '20090701', '20240630'
# write_path = '/scratch/ng72/ms5578/solar_wind_tseries/'

# # BARRA-R2 variable paths
# rsds_path = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/rsds/latest/"
# u_path    = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/ua50m/latest/"
# v_path    = "/g/data/ob53/BARRA2/output/reanalysis/AUS-11/BOM/ERA5/historical/hres/BARRA-R2/v1/day/va50m/latest/"

# # Generator site list
# gen_csv = "/g/data/ng72/ms5578/ID_HW_BARRA/data/preprocess/gen_info.csv"

# # Dask cluster
# client = Client()
# client

In [5]:
# # ============================
# # Functions
# # ============================
# def get_files(path, sdate, edate):
#     """Get sorted list of files in date range."""
#     fdates = [m.strftime('%Y%m') for m in pd.date_range(sdate, edate, freq='ME')]
#     fpaths = [s for s in os.listdir(path) if any(f in s for f in fdates)]
#     fpaths = sorted([os.path.join(path, f) for f in fpaths])
#     if not fpaths:
#         raise FileNotFoundError(f"No NetCDF files found for {sdate} to {edate} in {path}")
#     print(f"{path}: found {len(fpaths)} files")
#     return fpaths


# def extract_and_save(varname, fpaths, coords_df, out_file):
#     with xr.open_dataset(fpaths[0]) as ds:
#         all_lats = ds['lat'].values
#         all_lons = ds['lon'].values
#         if np.any(all_lons > 180) and np.any(coords_df['lon'] < 0):
#             coords_df['lon'] = coords_df['lon'] % 360

#     combined_df = []

#     ds = xr.open_mfdataset(fpaths, combine='by_coords', chunks={'time': 50}, parallel=True)  # move outside the loop

#     for _, row in coords_df.iterrows():
#         lat_idx = np.abs(all_lats - row['lat']).argmin()
#         lon_idx = np.abs(all_lons - row['lon']).argmin()

#         ts = ds[varname].isel(lat=lat_idx, lon=lon_idx).to_series().reset_index()
#         ts['DUID'] = row['DUID']

#         combined_df.append(ts)

#     combined_df = pd.concat(combined_df, ignore_index=True)
#     combined_df.to_csv(out_file, index=False)
#     print(f"Saved combined {varname} data for all sites to {out_file}")


# def extract_wind(u_paths, v_paths, coords_df, out_file):
#     """
#     Extract ua50m and va50m for all sites, combine into one DataFrame,
#     add DUID column, and save as a single CSV file.
#     """
#     with xr.open_dataset(u_paths[0]) as ds:
#         all_lats = ds['lat'].values
#         all_lons = ds['lon'].values
#         if np.any(all_lons > 180) and np.any(coords_df['lon'] < 0):
#             coords_df['lon'] = coords_df['lon'] % 360

#     combined_df = []

#     for _, row in coords_df.iterrows():
#         lat_idx = np.abs(all_lats - row['lat']).argmin()
#         lon_idx = np.abs(all_lons - row['lon']).argmin()

#         u_ds = xr.open_mfdataset(u_paths, combine='by_coords', chunks={'time': 50}, parallel=True)
#         v_ds = xr.open_mfdataset(v_paths, combine='by_coords', chunks={'time': 50}, parallel=True)

#         u_ts = u_ds['ua50m'].isel(lat=lat_idx, lon=lon_idx).to_series()
#         v_ts = v_ds['va50m'].isel(lat=lat_idx, lon=lon_idx).to_series()

#         df = pd.DataFrame({
#             'time': u_ts.index,
#             'ua50m': u_ts.values,
#             'va50m': v_ts.values
#         })

#         df['DUID'] = row['DUID']
#         combined_df.append(df)

#     combined_df = pd.concat(combined_df, ignore_index=True)
#     combined_df.to_csv(out_file, index=False)
#     print(f"Saved combined wind data for all sites to {out_file}")


# # ============================
# # Main
# # ============================
# if __name__ == "__main__":
#     gen_df = pd.read_csv(gen_csv)

#     solar_df = gen_df[gen_df['fuel_source_primary'] == 'Solar'][['DUID', 'lat', 'lon']].copy()
#     wind_df  = gen_df[gen_df['fuel_source_primary'] == 'Wind'][['DUID', 'lat', 'lon']].copy()

#     rsds_files = get_files(rsds_path, sdate, edate)
#     u_files    = get_files(u_path, sdate, edate)
#     v_files    = get_files(v_path, sdate, edate)

#     solar_out_file = os.path.join(write_path, "rsds_solar.csv")
#     wind_out_file  = os.path.join(write_path, "wind.csv")
